# 01 — Synthetic Conversation Data Generation

The Edge Gatekeeper needs labelled conversational text. Real ambient-conversation
data is private and hard to obtain, so this notebook generates a realistic synthetic
dataset using a template + slot-filling strategy, then adds **ASR-style noise**
(lowercasing, dropped punctuation, spoken fillers) so the training distribution
looks like real speech-to-text output rather than clean written text.

### Label taxonomy (6 classes)
| label | meaning | gate policy |
|---|---|---|
| `question`  | questions that may need assistance | FORWARD |
| `task`      | tasks, commitments, follow-ups | FORWARD |
| `decision`  | decisions or changing plans | FORWARD |
| `info`      | important information or updates | FORWARD |
| `risk`      | warnings, risks, unusual situations | FORWARD |
| `ordinary`  | small talk, fillers, chit-chat | REJECT |

The 5 "meaningful" classes map directly to the moment types listed .
 Output: `data/train.csv`, `data/test.csv`,
`data/eval_scenarios.json` (hand-written conversations used by the simulator
and the evaluation report).

In [1]:
import json
import random
import re
from pathlib import Path

import pandas as pd

random.seed(42)

DATA_DIR = Path("..") / "data"
DATA_DIR.mkdir(exist_ok=True)

## 1. Slot vocabulary
Slots keep the templates compact while producing thousands of distinct surface forms.

In [2]:
SLOTS = {
    "person": ["mom", "dad", "priya", "rahul", "the landlord", "my boss", "dr. mehta",
               "the electrician", "aunt seema", "my brother", "the delivery guy",
               "sarah", "the teacher", "our neighbour", "the plumber", "ankit"],
    "place": ["the pharmacy", "the bank", "the school", "the office", "the airport",
              "the clinic", "the metro station", "the grocery store", "the gym",
              "the restaurant", "the service center", "the market"],
    "thing": ["the report", "the invoice", "the documents", "the laptop", "the keys",
              "the medicine", "the tickets", "the presentation", "the form",
              "the charger", "the parcel", "the rent", "the assignment", "the photos"],
    "day": ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday",
            "sunday", "tomorrow", "next week", "this evening", "tonight",
            "day after tomorrow", "this weekend"],
    "time": ["9 am", "10:30", "noon", "4 pm", "5:30", "7 in the evening", "8 pm",
             "early morning", "lunchtime"],
    "event": ["the meeting", "the appointment", "the wedding", "the interview",
              "the class", "the call", "the trip", "the party", "the checkup",
              "the flight", "the review"],
    "chore": ["call the plumber", "pay the electricity bill", "book the tickets",
              "pick up the medicine", "send the email", "renew the insurance",
              "submit the form", "collect the laundry", "recharge the phone",
              "water the plants", "return the library books", "fix the tap"],
    "topic": ["the weather", "the match", "that movie", "the food", "the traffic",
              "the music", "that show", "cricket", "the news"],
    "hazard": ["the stove is still on", "the gas smells weird", "the door is unlocked",
               "the floor is wet", "the wire is sparking", "the brakes feel loose",
               "the balcony railing is loose", "the milk has gone bad",
               "the iron was left plugged in", "the tap is leaking badly"],
    "update": ["the flight got delayed", "the school is closed", "the meeting got cancelled",
               "the results are out", "the prices went up again", "the road is blocked",
               "the power will be cut", "the train is running late",
               "the office is shifting to the new building", "the exam got postponed"],
    "amount": ["500 rupees", "2000 rupees", "a lot", "half of it", "the full amount",
               "1200", "more than expected"],
}


def fill(template: str) -> str:
    """Fill {slot} placeholders with random vocabulary."""
    def repl(m):
        return random.choice(SLOTS[m.group(1)])
    return re.sub(r"\{(\w+)\}", repl, template)

## 2. Templates per class

In [3]:
TEMPLATES = {
    "question": [
        "do you know what time {place} closes",
        "can you check when {event} starts",
        "what time does {place} open on {day}",
        "does anyone know where i kept {thing}",
        "how do we get to {place} from here",
        "can someone find out if {place} is open {day}",
        "do you know how much the tickets cost",
        "is there a way to reschedule {event}",
        "what was the address {person} sent",
        "how long does it take to reach {place}",
        "do we need to carry {thing} for {event}",
        "can you look up the number for {place}",
        "when is the last date to submit {thing}",
        "which bus goes to {place}",
        "do you remember what {person} said about {event}",
        "is {place} open on {day} or not",
        "what should we do about {thing}",
        "how much did we pay for {thing} last time",
        "can you find a good deal on {thing}",
        "who should i contact about {thing}",
        "any idea when {person} is coming back",
        "where do we submit {thing}",
        "could you find out the fees for {event}",
        "what's the best way to reach {place} at {time}",
        "did anyone ask {person} about {thing}",
        "is it possible to change the date of {event}",
        "what documents do they need at {place}",
        "how many people are coming to {event}",
        "should we take a cab or the metro to {place}",
        "whom do we call if {place} is shut",
        "can somebody check the status of {thing}",
        "what did the doctor say about the reports",
        "is the offer on {thing} still valid",
    ],
    "task": [
        "i'll send {thing} by {day}",
        "remind me to {chore} {day}",
        "i need to {chore} before {time}",
        "can you {chore} when you get time",
        "don't forget to {chore}",
        "i promised {person} i'd share {thing} by {day}",
        "we still have to {chore}",
        "put it on my list to {chore}",
        "i'll call {person} {day} about {thing}",
        "make sure {thing} reaches {person} by {time}",
        "i have to drop {thing} at {place} {day}",
        "let me finish {thing} tonight and send it over",
        "i'll follow up with {person} about {event}",
        "add {chore} to the todo list",
        "you take care of {thing} and i'll handle the rest",
        "i owe {person} {amount} i should pay it back {day}",
        "i told {person} we'd confirm by {time}",
        "need to book a cab for {time}",
        "i'll pick up {thing} from {place} on my way back",
        "have to submit {thing} before {event}",
        "note it down we must {chore} by {day}",
        "i'm supposed to send {thing} to {person} today",
        "keep {thing} ready i'll collect it at {time}",
        "i said i'd help {person} with {thing} {day}",
        "first thing {day} morning i'll {chore}",
        "someone has to {chore} before {event}",
        "i'll get it done and update you by {time}",
        "pending item is to {chore} don't let me forget",
        "i will share the details with {person} after {event}",
        "let me set a reminder to {chore}",
        "task for {day} is to {chore}",
        "i'll transfer {amount} to {person} by {day}",
        "gotta drop {person} at {place} at {time}",
        "still have to confirm the booking with {place}",
    ],
    "decision": [
        "let's move {event} to {day}",
        "we decided to cancel {event}",
        "okay final we're going with the cheaper option",
        "let's do {event} at {time} instead",
        "change of plan we're meeting at {place} now",
        "i've decided to quit and look for something better",
        "we're postponing {event} to {day}",
        "let's split the cost of {thing}",
        "fine let's not invite {person} then",
        "we'll take the morning train instead of the flight",
        "let's finalize {place} for {event}",
        "okay then {day} {time} is locked for {event}",
        "we are shifting {event} from {day} to {day}",
        "instead of buying let's just rent {thing}",
        "let's drop the idea of {event} this month",
        "we picked the second apartment the one near {place}",
        "so it's decided {person} will handle {thing}",
        "let's cut the budget for {thing} by {amount}",
        "we're switching to the new plan from {day}",
        "actually let's meet at {time} not {time}",
        "we agreed to keep {event} at {place} itself",
        "final call we're skipping {event} this time",
        "let's go ahead with {person}'s suggestion",
        "we chose the {time} slot for {event}",
        "plan is fixed now {place} on {day}",
        "let's cancel the cab and take the train",
        "it's settled {person} pays for {thing} this time",
        "we're not renewing the subscription anymore",
        "scrap the old plan we start fresh from {day}",
        "moving {thing} to {day} everyone okay with that done then",
        "we'll do {event} first and {chore} after",
        "let's book {place} before the prices go up",
        "decided we're selling the old bike",
        "no more debate we go with option two",
    ],
    "info": [
        "{update} {day}",
        "just heard that {update}",
        "{person} said {update}",
        "fyi {update} so plan accordingly",
        "got a message that {update}",
        "{person} confirmed {event} is at {time}",
        "the new timings for {place} are {time} to {time}",
        "{person} got selected for the job",
        "the doctor changed the dosage of the medicine",
        "our booking at {place} is confirmed for {day}",
        "{person} reached safely an hour ago",
        "the refund of {amount} got credited today",
        "{event} venue changed to {place}",
        "{person} is coming down {day} for {event}",
        "the landlord increased the rent by {amount}",
        "results came and she passed with distinction",
        "{thing} finally got delivered to {place}",
        "the deadline for {thing} moved to {day}",
        "{person} called to say {event} starts at {time} sharp",
        "insurance claim got approved for {amount}",
        "heads up {event} has been preponed to {time}",
        "update {person} cleared the interview",
        "they announced a holiday on {day}",
        "{place} is running a big sale till {day}",
        "the salary got credited early this month",
        "{person} shifted to the new place last week",
        "the maintenance charges went up by {amount}",
        "got the confirmation {thing} ships {day}",
        "the strike is over buses are back from {day}",
        "{person} texted that {event} is happening at {place}",
        "our slot at {place} got moved to {time}",
        "news says {place} metro line opens {day}",
        "the warranty on {thing} covers the repair",
        "{person} is getting discharged {day} morning",
    ],
    "risk": [
        "i think {hazard}",
        "careful {hazard}",
        "hey {hazard} someone should check",
        "did you notice {hazard}",
        "{person} said {hazard} please look at it",
        "watch out {hazard}",
        "i'm worried {person} sounded really unwell on the call",
        "the car is making a strange noise again",
        "{person}'s fever isn't coming down since {day}",
        "there's water leaking near the switchboard",
        "somebody left the gate open all night",
        "the medicine expired last month don't take it",
        "i smell something burning from the kitchen",
        "the bike's tyre looks completely worn out",
        "grandpa felt dizzy again this morning",
        "that link {person} forwarded looks like a scam",
        "someone tried to open the door around {time} last night",
        "the internet banking otp came without me doing anything",
        "the ladder is shaky don't climb it alone",
        "kids were playing near the main road again",
        "don't eat that it's been out since {day}",
        "there's a crack on the wall near {place} getting bigger",
        "the geyser was on the whole night that's risky",
        "someone was following me from {place} it felt off",
        "the inverter battery is swelling up don't touch it",
        "{person} hasn't picked up the phone since {time}",
        "the ceiling fan is wobbling badly get it checked",
        "low balance alert came twice something's wrong with the account",
        "the pressure cooker whistle is stuck be careful",
        "street dogs near {place} looked aggressive today",
        "the extension board is overheating unplug it",
        "there's a weird charge of {amount} on the card statement",
        "the lift stopped between floors again avoid it",
        "roads near {place} are flooded don't take that route",
    ],
    "ordinary": [
        "yeah",
        "hmm okay",
        "haha that's funny",
        "nice weather today",
        "what's up",
        "i'm fine thanks",
        "did you watch {topic} yesterday",
        "the food was really good",
        "lol",
        "yeah i know right",
        "so tired today man",
        "this song is stuck in my head",
        "the traffic was crazy as usual",
        "anyway how was your day",
        "nothing much just chilling",
        "that was a great match",
        "i love this place",
        "hahaha good one",
        "hmm let me think",
        "yeah yeah totally",
        "it's so hot these days",
        "we should hang out sometime",
        "long time no see",
        "you look great today",
        "oh nice",
        "cool cool",
        "same here",
        "i had biryani for lunch",
        "that actor is so good in {topic}",
        "my phone battery dies so fast",
        "how are you doing",
        "good morning everyone",
        "okay bye see you",
        "talk later",
        "true true",
        "no way really",
        "just scrolling reels",
        "the coffee here is decent",
        "what a boring day",
        "acha okay",
        "this tea is perfect",
        "did you see how crowded {place} was",
        "hmm yeah makes sense",
        "he's always late anyway",
        "i slept so well last night",
        "the new phone looks nice though",
        "arre wah nice",
        "kids these days i tell you",
        "that meme was hilarious",
        "monsoon started early this year",
        "i miss home food",
        "she sings really well",
        "the gym was empty today",
        "nothing interesting on tv these days",
        "my back hurts from sitting all day",
        "the dog did the funniest thing today",
        "weekend went by so fast",
        "yeah i heard that song",
    ],
}

## 3. ASR-style noise
Wearable speech-to-text output is lowercase-ish, lightly punctuated, and full of
spoken fillers. Injecting that noise at generation time keeps the training
distribution close to what the Gatekeeper will actually see.

In [4]:
FILLERS_PRE = ["um ", "uh ", "so ", "hey ", "listen ", "arre ", "okay so ", "actually ",
               "you know ", "by the way "]
FILLERS_POST = [" na", " yaar", " right", " you know", " actually", " only"]


def asr_noise(text: str) -> str:
    t = text.lower()
    t = re.sub(r"[^\w\s']", "", t)              # ASR rarely emits punctuation
    if random.random() < 0.35:
        t = random.choice(FILLERS_PRE) + t
    if random.random() < 0.18:
        t = t + random.choice(FILLERS_POST)
    if random.random() < 0.08:                   # occasional word repetition (dysfluency)
        words = t.split()
        if len(words) > 2:
            i = random.randrange(len(words) - 1)
            words.insert(i, words[i])
            t = " ".join(words)
    return t.strip()

## 4. Generate, deduplicate, split

In [5]:
PER_CLASS = 1400
rows = []
for label, templates in TEMPLATES.items():
    seen = set()
    attempts = 0
    while len(seen) < PER_CLASS and attempts < PER_CLASS * 60:
        attempts += 1
        tmpl_id = random.randrange(len(templates))
        utterance = asr_noise(fill(templates[tmpl_id]))
        if utterance not in seen and len(utterance) > 1:
            seen.add(utterance)
            rows.append({"text": utterance, "label": label,
                         "template": f"{label}_{tmpl_id}"})

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(df["label"].value_counts())
print(f"\ntotal: {len(df)} utterances")
df.head(10)

label
question    1400
info        1400
decision    1400
task        1400
ordinary    1400
risk        1400
Name: count, dtype: int64

total: 8400 utterances


,text,label,template
0,where do we submit the invoice,question,question_21
1,insurance claim got approved approved for 2000...,info,info_19
2,our booking at the market is confirmed for nex...,info,info_9
3,let's move the meeting to tonight,decision,decision_0
4,by the way heads up the review has been prepon...,info,info_20
5,uh fyi the exam got postponed so plan accordingly,info,info_3
6,i will share the details with the delivery guy...,task,task_28
7,um any idea when the delivery guy is coming back,question,question_20
8,the delivery guy got selected for the job,info,info_7
9,listen how much did we pay for the parcel last...,question,question_17


### Template-disjoint split
A naive random split leaks: train and test would share templates, so any model
scores ~100% and the evaluation is meaningless. Instead, **~20% of the templates
in each class are held out entirely** — the test set contains only phrasings the
model has never seen. This measures generalisation to new wordings, which is
what actually happens on-device.

In [6]:
test_templates = set()
for label, templates in TEMPLATES.items():
    n_hold = max(2, round(len(templates) * 0.2))
    held = random.sample(range(len(templates)), n_hold)
    test_templates.update(f"{label}_{i}" for i in held)

test_df = df[df["template"].isin(test_templates)].drop(columns="template")
train_df = df[~df["template"].isin(test_templates)].drop(columns="template")
train_df.to_csv(DATA_DIR / "train.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)
print(f"train: {len(train_df)} (templates seen)   ->  data/train.csv")
print(f"test:  {len(test_df)} (templates UNSEEN) ->  data/test.csv")
print(test_df['label'].value_counts())

train: 6702 (templates seen)   ->  data/train.csv
test:  1698 (templates UNSEEN) ->  data/test.csv
label
task        307
decision    305
question    297
risk        280
ordinary    263
info        246
Name: count, dtype: int64


## 4b. Hedge / vague-referent training augmentation

The scenario suite below (§5) intentionally includes lines where a confident
decision is *not* desirable — vague unresolved-referent statements ("we
should do something about **it**") and hedged doubt about a plan ("i'm not
sure that works"). Evaluation showed the base classifier gets these
overconfident, in **both** directions:

- unresolved-action lines get overconfidently `FORWARD`ed (the model sees
  strong cue words but not that the referent is unresolved);
- hedged doubt about a plan gets overconfidently `REJECT`ed (hedge tokens
  dominate and the model has no representation of disagreement-about-a-plan
  as meaningful).

This cell adds a **train-only** augmentation set — never mixed into the
per-class template pool above, so the template-disjoint test split and its
metrics stay exactly as they were. Each pattern is treated according to the
*direction* of its baseline error:

- **task** (unresolved action) was overconfidently forwarded, so its
  phrases are **dual-labelled** — emitted under both `task` and `ordinary` —
  which teaches the model this region sits nearer p(meaningful)≈0.5.
- **decision** (hedged doubt about a plan) was overconfidently *rejected*,
  the opposite problem, so it gets **positive-only** augmentation (no
  `ordinary` pairing) — pure additional signal that this pattern is
  plan-adjacent, without a counteracting pull toward `ordinary`.

A third category, **vague future risk** ("that might be a problem later"),
was tried the same way as `task` and is deliberately **not** included below:
dual-labelling it nudged the corresponding scenario line from an
overconfident `FORWARD` (p=0.68 — the *safe* error, still forwarded, just
without the low-priority flag) into a confident `REJECT` (p=0.29 — the
*unsafe* error, the moment would actually be dropped). Per this project's
own error-cost framing (`docs/evaluation_report.md` §4), trading a safe-
direction miss for an unsafe-direction one is a net loss even though the
scenario table alone would show it as "no longer overconfident FORWARD" —
so it was left out rather than silently accepting that trade. This remains
an open item for future work: fixing vague, referent-free risk phrasing
properly likely needs real examples of the pattern rather than more
synthetic guessing.

None of the remaining phrases are copies of the (independently hand-written)
sentences in `data/eval_scenarios.json` — the referent is deliberately kept
bare ("it"/"this"/"that", never slot-filled into a concrete noun), because
that unresolved-referent shape is exactly the feature being taught, and the
scenario suite must stay an untouched generalisation check, same as the rest
of this dataset.

In [7]:
AMBIGUOUS = {
    # vague unresolved action -- sounds task-like but the referent is bare.
    # Baseline model was overconfident FORWARD here (p=0.96 on the scenario
    # line); dual-labelling with "ordinary" pulls this region toward the band.
    "task": {
        "dual_label": True,
        "noise_variants": 10,
        "phrases": [
            "someone needs to sort this out",
            "we'll deal with it eventually",
            "i guess we'll figure something out",
            "somebody should look into that soon",
            "we really need to address this at some point",
            "we'll get around to it i suppose",
            "someone ought to take care of that",
            "we should probably fix this sometime",
            "not sure who's handling that but someone should",
            "we'll attend to it later i guess",
            "somebody has to deal with this at some point",
            "we'll sort it out one of these days",
        ],
    },
    # hedged doubt/disagreement about a plan -- baseline was overconfident
    # REJECT (p=0.16, the opposite direction). The evaluation report's own
    # diagnosis of this failure mode is "no representation of disagreement-
    # about-a-plan as meaningful", so this category gets pure positive
    # signal only -- no "ordinary" pairing, unlike "task" above.
    "decision": {
        "dual_label": False,
        "noise_variants": 9,
        "phrases": [
            "i'm not sure that's going to work",
            "i don't think this is a good idea",
            "not convinced this plan works",
            "i have my doubts about this",
            "i don't know if that's going to hold up",
            "something about this doesn't feel right",
            "i'm not totally sure this is the right call",
            "i doubt that's going to go smoothly",
            "not sure this is going to pan out",
            "i'm skeptical this is going to work out",
            "i'm second-guessing this whole plan",
            "part of me thinks this won't work",
        ],
    },
    # NOTE: a "vague future risk" category (dual-labelled with "ordinary")
    # was tried here and dropped. It nudged a vague risk-ish scenario line
    # from an overconfident FORWARD (p=0.68, the *safe* error direction --
    # still forwarded, just without the low-priority flag) into a confident
    # REJECT (p=0.29, the *unsafe* direction -- the moment would actually be
    # dropped). Per the project's own error-cost framing, a false reject is
    # unrecoverable while an over-forward only wastes downstream compute, so
    # trading one for the other is a net loss even though it "looks" like a
    # fix in isolation. Left out rather than risk that trade silently.
}

amb_rows = []
seen_amb = set()
for label, cfg in AMBIGUOUS.items():
    for phrase in cfg["phrases"]:
        for _ in range(cfg["noise_variants"]):
            utterance = asr_noise(phrase)
            key = (label, utterance)
            if utterance and key not in seen_amb:
                seen_amb.add(key)
                amb_rows.append({"text": utterance, "label": label})
                if cfg["dual_label"]:
                    amb_rows.append({"text": utterance, "label": "ordinary"})

amb_df = (pd.DataFrame(amb_rows)
          .drop_duplicates()
          .sample(frac=1, random_state=42)
          .reset_index(drop=True))
print(f"ambiguous augmentation: {len(amb_df)} rows from "
      f"{sum(len(c['phrases']) for c in AMBIGUOUS.values())} base phrases")
print(amb_df["label"].value_counts())

train_df = (pd.concat([train_df, amb_df], ignore_index=True)
            .sample(frac=1, random_state=42)
            .reset_index(drop=True))
train_df.to_csv(DATA_DIR / "train.csv", index=False)
print(f"\ntrain (base + ambiguous augmentation): {len(train_df)} rows -> data/train.csv")

ambiguous augmentation: 199 rows from 24 base phrases
label
task        69
ordinary    69
decision    61
Name: count, dtype: int64

train (base + ambiguous augmentation): 6901 rows -> data/train.csv


## 5. Hand-written evaluation scenarios
The generated data tests the classifier; these **hand-written conversations**
test the *whole Gatekeeper* (incremental context, duplicates, uncertainty).
They deliberately include easy cases, near-duplicates, and ambiguous lines,
and they are used by both the simulator UI and the evaluation report.

In [8]:
EVAL_SCENARIOS = [
    {
        "name": "Morning at home",
        "description": "Household chatter with a safety risk, a task and a repeated line.",
        "turns": [
            {"text": "good morning", "expected": "REJECT"},
            {"text": "slept okay i guess", "expected": "REJECT"},
            {"text": "hey i think the stove is still on", "expected": "FORWARD"},
            {"text": "remind me to pay the electricity bill today", "expected": "FORWARD"},
            {"text": "the stove is still on someone check", "expected": "DUPLICATE"},
            {"text": "it's so hot these days", "expected": "REJECT"},
            {"text": "mom said the school is closed tomorrow", "expected": "FORWARD"},
        ],
    },
    {
        "name": "Office corridor",
        "description": "Work small talk that turns into a decision and a commitment.",
        "turns": [
            {"text": "did you watch the match yesterday", "expected": "REJECT"},
            {"text": "haha yeah what a game", "expected": "REJECT"},
            {"text": "okay so let's move the client meeting to thursday", "expected": "FORWARD"},
            {"text": "i'll send the revised deck by tonight", "expected": "FORWARD"},
            {"text": "cool cool", "expected": "REJECT"},
            {"text": "also fyi the office is shifting to the new building next month", "expected": "FORWARD"},
        ],
    },
    {
        "name": "Ambiguous moments",
        "description": "Lines where a confident decision is NOT desirable — the Gatekeeper should be uncertain.",
        "turns": [
            {"text": "we should do something about it", "expected": "UNCERTAIN"},
            {"text": "that might be a problem later", "expected": "UNCERTAIN"},
            {"text": "hmm i'm not sure that works", "expected": "UNCERTAIN"},
            {"text": "yeah maybe", "expected": "REJECT"},
        ],
    },
    {
        "name": "Evening plans",
        "description": "Plan changes, a question, and repeated confirmations.",
        "turns": [
            {"text": "so tired today man", "expected": "REJECT"},
            {"text": "do you know what time the pharmacy closes", "expected": "FORWARD"},
            {"text": "change of plan we're meeting at the cafe now", "expected": "FORWARD"},
            {"text": "we're meeting at the cafe now okay", "expected": "DUPLICATE"},
            {"text": "okay bye see you", "expected": "REJECT"},
        ],
    },
]

with open(DATA_DIR / "eval_scenarios.json", "w") as f:
    json.dump(EVAL_SCENARIOS, f, indent=2)
print("wrote data/eval_scenarios.json with", sum(len(s["turns"]) for s in EVAL_SCENARIOS), "turns")

wrote data/eval_scenarios.json with 22 turns
